# 🛡️ PharmaGuard — AI Pharmacy Security Monitor
### Cloud-Assisted Real-Time Suspicious-Interaction Monitoring System

---

## 1. Introduction

**PharmaGuard** is an AI-assisted real-time surveillance system designed for pharmacy operations. It monitors designated pharmacy shelf areas (e.g. prescription medicines or controlled substances), tracks individuals across frames, detects suspicious behavioral interaction sequences (such as prolonged dwell/loitering or rapid grab-and-retreat movement), and alerts pharmacy staff for human review.

### Cloud-Assisted Architecture
The system employs a browser-to-cloud computer vision pipeline:
```
[ Laptop Webcam ]
        │  (browser captures 640x480 JPEG frames, one in flight at a time)
        ▼
[ Secure HTTPS Live Tunnel ] (Gradio .live tunnel)
        │
        ▼
[ Google Colab GPU Cloud Runtime ]
        ├─► [ Person Detection ]  (Pretrained YOLOv8 / RT-DETR)
        ├─► [ Multi-Object MOT ]  (IoU tracker, ByteTrack-style)
        ├─► [ Behavior Analysis ] (Shelf ROI Dwell Time & Loitering)
        ├─► [ Alert Management ]  (Severity Grading & Cooldown Throttling)
        └─► [ Frame Annotation ]  (Bounding Boxes, IDs, Trails & HUD)
        │
        ▼
[ Browser Monitoring Dashboard ] (Real-Time Video + Telemetry + Alerts)
```


---

## 2. Runtime Configuration

Inspect the active Colab environment, Python runtime version, CUDA drivers, and GPU hardware allocation. A GPU runtime (NVIDIA T4, L4, V100, or A100) is strongly recommended for real-time inference.


In [ ]:
import sys
import platform
import torch

print("=" * 65)
print("🔍 PHARMAGUARD RUNTIME ENVIRONMENT CHECK")
print("=" * 65)

# 1. Python Environment
print(f"Python Version:   {platform.python_version()} ({sys.executable})")

# 2. PyTorch & CUDA Availability
print(f"PyTorch Version:  {torch.__version__}")
cuda_available = torch.cuda.is_available()
print(f"CUDA Available:   {cuda_available}")

if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_count = torch.cuda.device_count()
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    cuda_version = torch.version.cuda
    print(f"Active GPU Model: {gpu_name} ({total_mem_gb:.2f} GB VRAM)")
    print(f"Allocated GPUs:   {gpu_count}")
    print(f"CUDA Toolkit:     {cuda_version}")
    print("\n✅ GPU hardware detected. Real-time inference acceleration enabled.")
else:
    print("\n⚠️  WARNING: NO GPU DETECTED!")
    print("   Running on CPU will limit processing throughput to ~8-10 FPS.")
    print("   To switch to GPU: Click Runtime > Change runtime type > Select T4 GPU > Save.")
print("=" * 65)


---

## 3. Repository Setup

Clone the official PharmaGuard GitHub repository into `/content/pharmguard` and verify that all core modules exist. If the directory already exists, it pulls the latest updates from remote.


In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/Franklinnn099/Pharmguard.git"
TARGET_DIR = "/content/pharmguard"

print("=" * 65)
print("📁 PHARMAGUARD REPOSITORY INITIALIZATION")
print("=" * 65)

if os.path.exists(TARGET_DIR):
    print(f"Repository directory already exists at {TARGET_DIR}.")
    print("Pulling latest commits from GitHub...")
    os.chdir(TARGET_DIR)
    res = subprocess.run(["git", "pull"], capture_output=True, text=True, check=False)
    print(res.stdout.strip() or res.stderr.strip())
elif os.path.exists("pharmaguard") and os.path.exists("main.py"):
    print(f"Already running from project directory: {os.getcwd()}")
else:
    print(f"Cloning repository from {REPO_URL} into {TARGET_DIR}...")
    res = subprocess.run(["git", "clone", REPO_URL, TARGET_DIR], capture_output=True, text=True, check=False)
    if res.returncode == 0:
        print("Repository cloned successfully.")
    else:
        print(f"Clone error: {res.stderr}")
    os.chdir(TARGET_DIR)

# Verify presence of all required project files
required_files = [
    "main.py",
    "benchmark.py",
    "requirements.txt",
    "pharmaguard/__init__.py",
    "pharmaguard/config.py",
    "pharmaguard/detector.py",
    "pharmaguard/tracker.py",
    "pharmaguard/behavior.py",
    "pharmaguard/alerts.py",
    "pharmaguard/events.py",
    "pharmaguard/annotator.py",
    "pharmaguard/pipeline.py",
    "pharmaguard/ui.py",
    "pharmaguard/app.py",
]

missing_files = [f for f in required_files if not os.path.exists(f)]
if missing_files:
    print(f"❌ Warning: Missing required project files: {missing_files}")
else:
    print(f"✅ All {len(required_files)} core modules and runners verified in {os.getcwd()}.")
print("=" * 65)


---

## 4. Dependency Installation

Install the pinned dependencies from `requirements.txt`.
- Handles package installations cleanly without arbitrary version changes.
- Checks compatibility of Gradio and Pydantic schemas to ensure smooth WebSocket streaming.
- Verifies critical vision and machine learning imports.


In [ ]:
import sys
import subprocess

print("=" * 65)
print("📦 INSTALLING PHARMAGUARD DEPENDENCIES")
print("=" * 65)

# Install pinned dependencies
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# Verify critical package imports and versions
import torch
import cv2
import ultralytics
import gradio as gr
import pydantic
import numpy as np

print("\nPackage Verification:")
print(f"  - PyTorch:      {torch.__version__}")
print(f"  - OpenCV:       {cv2.__version__}")
print(f"  - Ultralytics:  {ultralytics.__version__}")
print(f"  - Gradio:       {gr.__version__}")
print(f"  - Pydantic:     {pydantic.__version__}")
print(f"  - NumPy:        {np.__version__}")

# Check Gradio / Pydantic JSON-schema compatibility
try:
    import jsonschema
    print(f"  - JSONSchema:   {jsonschema.__version__}")
    print("✅ Schema serialization verified clean.")
except ImportError:
    print("ℹ️ jsonschema module optional; default validation active.")

print("=" * 65)


---

## 5. Authentication Configuration

PharmaGuard requires authentication on public tunnels to prevent unauthorized viewing of the pharmacy surveillance feed.

### How Authentication Works:
1. **Google Colab Secrets (Recommended)**: Set `PHARMAGUARD_AUTH_USER` and `PHARMAGUARD_AUTH_PASS` in Colab's left sidebar (🔑 Secrets icon) and toggle "Notebook access".
2. **Environment Variables**: Configure your desired username and password in the cell below.
3. **Automatic Ephemeral Token**: If no credentials are set, `pharmaguard/app.py` will generate a secure, random one-time password and display it upon launch.


In [ ]:
import os
import getpass

print("=" * 65)
print("🔐 CONFIGURE DASHBOARD AUTHENTICATION")
print("=" * 65)

# 1. Attempt retrieval from Google Colab Secrets
colab_user, colab_pass = None, None
try:
    from google.colab import userdata
    colab_user = userdata.get("PHARMAGUARD_AUTH_USER")
    colab_pass = userdata.get("PHARMAGUARD_AUTH_PASS")
except Exception:
    pass

if colab_user and colab_pass:
    os.environ["PHARMAGUARD_AUTH_USER"] = colab_user
    os.environ["PHARMAGUARD_AUTH_PASS"] = colab_pass
    print(f"✅ Loaded credentials securely from Colab Secrets (Operator: {colab_user})")
elif os.environ.get("PHARMAGUARD_AUTH_USER") and os.environ.get("PHARMAGUARD_AUTH_PASS"):
    print(f"✅ Loaded credentials from environment (Operator: {os.environ.get('PHARMAGUARD_AUTH_USER')})")
else:
    print("ℹ️  No static credentials configured.")
    print("   PharmaGuard will generate a one-time ephemeral session password on startup.")
    print("   (Optional: to configure your own credentials, set PHARMAGUARD_AUTH_USER and PHARMAGUARD_AUTH_PASS)")
print("=" * 65)


---

## 6. Model Initialization & Verification

Preload the pretrained model weights (`yolov8n-pose.pt` for people + wrist keypoints; `yolov8n.pt` is also used for bag detection), verify that the model is allocated to the GPU, and run an initial warmup inference on an empty test frame.


In [ ]:
import torch
import numpy as np
from pharmaguard.config import DetectorConfig
from pharmaguard.detector import YOLODetector

print("=" * 65)
print("🤖 INITIALIZING VISION DETECTOR")
print("=" * 65)

target_device = "cuda" if torch.cuda.is_available() else "cpu"
detector_cfg = DetectorConfig(model_name="yolov8n-pose.pt", device=target_device)  # pose: person boxes + wrist/hip keypoints

print(f"Loading {detector_cfg.model_name} onto compute device: {target_device.upper()}...")
detector = YOLODetector(detector_cfg)

model_info = detector.get_model_info()
print("Model Metadata:")
print(f"  - Weights File:        {model_info.get('model_name')}")
print(f"  - Active Compute Unit: {model_info.get('device').upper()}")
print(f"  - Confidence Filter:   {model_info.get('confidence_threshold')}")
print(f"  - Target Classes:      {model_info.get('target_classes')} (Person)")
print(f"  - Pose keypoints:      {model_info.get('has_keypoints')}")

# Warmup inference
dummy_frame = np.zeros((480, 640, 3), dtype=np.uint8)
test_output = detector.detect(dummy_frame)
print(f"Warmup inference completed successfully (Detections: {test_output.count}).")
print("✅ Model loaded and ready for real-time tracking.")
print("=" * 65)


---

## 7. GPU Benchmark (server compute only)

Runs `benchmark.py` on a real photo containing people (or a video via `--video`), measuring every server-side stage
**including the JPEG return encoding** Gradio performs after the pipeline:

`downscale -> detect -> track -> behaviour -> annotate -> encode`

> ⚠️ This is compute only. It does **not** include the browser <-> Colab network path, which is usually the larger cost
> over a `gradio.live` tunnel. Measure that from your laptop with `tools/e2e_benchmark.py` (see Section 8b),
> and read the live **Delivered FPS** / **Latency breakdown** panels on the dashboard.
>
> Record the printed table in `PROJECT_STATUS.md` - these are the first real GPU numbers for the project.

In [ ]:
# Shell (!) form streams output live into this cell.
!python benchmark.py --frames 100
# Faster variant if FPS is too low:  !python benchmark.py --frames 100 --imgsz 480 --max-width 480

---

## 7b. Persistent Event Storage on Google Drive (recommended)

Mount Drive so recorded events (SQLite) and evidence clips survive the Colab session.
PharmaGuard never opens the database directly on Drive (SQLite + Drive sync = locking/corruption risk):
it writes locally and copies to `MyDrive/PharmaGuard/` with SQLite's backup API.

> 🔒 Evidence clips show people's faces. Use staged footage of consenting participants only.
> Clips and events older than `recording.retention_days` (default 7) are deleted at startup.

In [ ]:
import os
MIRROR_ARG = ""
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/PharmaGuard", exist_ok=True)
    MIRROR_ARG = "--mirror-dir /content/drive/MyDrive/PharmaGuard"
    print("✅ Drive mounted: events will be mirrored to MyDrive/PharmaGuard")
except Exception as exc:
    print(f"ℹ️  Drive not mounted ({exc}); events stay in /content/pg_data for this session only.")

---

## 8. Launch PharmaGuard Security Console

Start the live PharmaGuard surveillance server with public HTTPS tunneling enabled (`--colab`).

### Instructions to Connect Your Laptop Webcam:
1. Run the cell below.
2. Look for the public tunnel link printed in the output: **`Running on public URL: https://xxxx.gradio.live`**.
3. Click the link to open the dashboard in your laptop browser.
4. **Log in**:
   - If you configured credentials in Section 5, enter them.
   - Otherwise, copy the **`[SECURITY] Generated one-time ephemeral session credentials`** printed in the output below.
5. In the **Live monitor** tab, click **Start camera** (fast stream: several frames in flight, much smoother over the tunnel).
6. When prompted by your browser (**Allow access to your camera?**), click **Allow**.
   - **Display: Smooth** (default) shows your live camera at full speed with detections drawn on top; boxes trail fast movement by ~one round trip. **Server images** shows exact annotated frames (choppier).
   - Detections slow? Try **Frames in flight** 4.
   - The old Gradio webcam is under *Fallback* (collapsed). Never run both at once.
7. Annotated frames return to the right-hand panel. Read the **measured** figures on the dashboard:
   - **Delivered FPS** (end-to-end, measured on the server from frame arrival spacing)
   - **Displayed FPS** (counted in your browser)
   - **Latency breakdown**: full cycle p50/p95, server compute, transport+encode

> 🔒 **Privacy**: frames travel through Gradio's public share relay (`*.gradio.live`). Use only staged footage of
> people who have agreed to be filmed - never real pharmacy customers.

### 8b. Measure end-to-end performance from your laptop (optional, ~1 min)
Stop the browser webcam first (one stream at a time), then on your **laptop** in the repo folder run:
```
python tools/e2e_benchmark.py --url https://xxxx.gradio.live --user <user> --password <password>
```
It prints round-trip p50/p95, sustained FPS, and a per-stage split (upload / wait / download).


In [ ]:
# Shell (!) form: the public URL and any generated password stream into this cell's output.
# Edit configs/demo_shelf.yaml to outline your mock shelf (see docs/DEMO_STAGING.md).
# Events + evidence clips go to fast local disk (/content/pg_data); if Drive is mounted (Section 7b)
# the database and clips are also copied to Drive after every event.
!python main.py --colab --config configs/demo_shelf.yaml --data-dir /content/pg_data {MIRROR_ARG}
# Lower-latency variant: add  --imgsz 480 --max-width 480

---

## 9. Live System Verification Guide

Full scenario list, camera placement and expected results: **`docs/DEMO_STAGING.md`**.

| Test | Action | Expected result |
| :--- | :--- | :--- |
| **1. Stream up** | Start the webcam on the left. | Annotated frames appear; status `LIVE | CUDA`. |
| **2. People + IDs** | One, then two people enter. | Boxes with distinct `ID #n`; IDs survive crossing paths. |
| **3. Wrists** | Raise a hand. | Green dot on each visible wrist. |
| **3b. Zone** | **Zones** tab: Capture snapshot, click the shelf corners, Add zone, Apply, Save. | Zone appears on the live video. |
| **4. Shelf contact** | Reach an arm into the outlined shelf. | Wrist dot turns orange; zone badge shows `HAND CONTACT`. |
| **5. Browsing** | Reach in several times, look at items at chest height, put them back. | **No** review/alert. |
| **6. Staged concealment** | Reach in, then within 1-2 s put the hand into a handbag and walk away. | Amber REVIEW, then red ALARM + siren with reasons; Acknowledge silences, Reset clears. |
| **7. Alarm** | Click the page once, then **Test alarm**. | `Alarm sound: ready`; 5 s siren, purple banner/border. |
| **7b. Review** | **Event review** tab: open the alert, play the clip, mark Confirmed / False alarm. | Clip plays; false-alarm rate updates. |
| **8. Measured performance** | Stream 30 s. | Record Delivered FPS, Displayed FPS, cycle p50/p95, server ms. |
| **9. Disconnect** | Stop the webcam for 5 s. | `NO SIGNAL` within ~2 s, then recovers with `1 interruption(s)`. |

> Camera must view the actor from ~45° to the side so the reaching arm is visible (a camera directly behind the actor cannot see hands).

In [ ]:
# Record the environment this run used (paste into PROJECT_STATUS.md alongside your measurements).
import torch, gradio, ultralytics, platform
print("Python", platform.python_version(), "| torch", torch.__version__, "| CUDA", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("gradio", gradio.__version__, "| ultralytics", ultralytics.__version__)

---

## 10. Troubleshooting & Common Questions

### 1. GPU Unavailable (`CUDA Available: False`)
- **Cause**: Google Colab started in CPU runtime mode.
- **Solution**: From the Colab top menu, select **Runtime > Change runtime type > Hardware accelerator > T4 GPU** and click **Save**. Re-run from Section 2.

### 2. Gradio JSON-Schema Warning or Error
- **Cause**: Mismatched `pydantic` or `jsonschema` library versions in Colab.
- **Solution**: Run `!pip install -q --upgrade "pydantic>=2.5.0" "jsonschema>=4.19.0"` and restart the session (**Runtime > Restart session**).

### 3. Webcam Permission Denied in Browser
- **Cause**: Browser security policy or conflicting application.
- **Solution**: Modern browsers strictly require **HTTPS** for camera access. Always open the public `https://xxxx.gradio.live` link (never an unencrypted `http://` link). Ensure no other application (Zoom, Teams, Skype) is currently using your webcam.

### 4. Public Gradio Live Tunnel Disconnected
- **Cause**: Network timeout or Colab cellular/Wi-Fi interruption.
- **Solution**: Stop Section 8 by clicking the Stop button (or **Runtime > Interrupt execution**), wait 5 seconds, and re-run Section 8 to generate a fresh link.

### 5. Authentication Failure
- **Cause**: Incorrect username or password entered.
- **Solution**: Check the console output of Section 8 for the generated one-time credentials, or configure `PHARMAGUARD_AUTH_USER` and `PHARMAGUARD_AUTH_PASS` in Section 5.

### 6. Model Weight Download Error
- **Cause**: Intermittent network issue downloading `yolov8n.pt` from GitHub.
- **Solution**: Re-run Section 6 to re-trigger download.


---

## 11. Shutdown and Cleanup

When your surveillance demonstration is finished:

1. **Stop the Application**: Click the **Interrupt** (Stop) button on the code cell running Section 8, or press `Ctrl + M I` (or `Cmd + M I` on macOS).
2. **Release Colab GPU Resources**: To conserve your Google Colab compute units, disconnect the runtime from the top menu: **Runtime > Disconnect and delete runtime**.
